# Study Tutor — the retrieval (RAG) pipeline

**IS469 group project · Retrieval part** · everything from an uploaded file to the passages the tutor answers from.

This notebook walks through the pipeline in order and ends by **measuring** which chunking method works best
on our slide corpus. Each section explains a step, shows its code, and runs it.

```
upload ─► 1 extract ─► 2 clean ─► 3 chunk ─► 4 embed ─► 5 search ─► top-5 passages (with citations)
                                                                       │
                                               6 evaluate: 40 gold questions → Hit@5, Recall@5, MRR
```

| Step | Code (in `src/tutor/retrieval/`) | Course link |
|:--|:--|:--|
| 1 · Extract | `extract.py` | — |
| 2 · Clean | `clean.py` | Topic 1 (text normalisation) |
| 3 · Chunk | `chunk.py`: per-slide, fixed, recursive, semantic | Topic 8, slides 13, 19–22 |
| 4 · Embed | `embed.py` (`all-MiniLM-L6-v2`) | Topic 8, slides 13–16 |
| 5 · Search | `search.py` (cosine similarity) | Topic 8, slides 15, 17 |
| 6 · Evaluate | `eval/eval_retrieval.py` + `eval/gold/slides.csv` | Topic 8, slide 25; Topics 5–6 |

**There is one copy of the code**, in `src/` (the app and the agent import it from there). This notebook imports
it and uses `show(...)` to print the source of each function, so what you read here is always the real code.

## 0 · Setup
In VS Code: *Select Kernel → Python (IS-469 .venv)*.

In [1]:
import inspect
import sys
from collections import Counter
from pathlib import Path

import numpy as np
from IPython.display import Markdown, display

# The project folder: works whether the notebook is run from notebooks/ or the project root.
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path[:0] = [str(ROOT / "src"), str(ROOT / "eval")]

from tutor.retrieval import chunk, clean, embed as embed_module, extract, search, types
from tutor.retrieval.chunk import chunk_pages
from tutor.retrieval.clean import clean_pages
from tutor.retrieval.embed import Embedder
from tutor.retrieval.extract import extract as extract_file, file_sha256
from tutor.retrieval.search import DenseIndex


def show(*objects):
    """Print the source code of functions/classes straight from src/."""
    for obj in objects:
        display(Markdown(f"```python\n{inspect.getsource(obj)}```"))


CORPUS = ROOT / "corpora" / "slides"
FILES = sorted(p for p in CORPUS.iterdir()
               if p.suffix.lower() in (".pdf", ".pptx") and not p.name.startswith("~$"))
print("Corpus files:", [f.name for f in FILES])

Corpus files: ['Topic7_Agentic_IS469.pdf', 'Topic8_AgenticRAG_MITB.pptx']


### The records passed between steps
Every step hands the next one the same small records. `file` and `page` travel with the text all the way
through, which is what makes citations like *"Topic8.pptx, slide 20"* possible.

| Record | Made by | Holds |
|:--|:--|:--|
| `PageRecord` | extract | the text of one PDF page or one slide |
| `Chunk` | chunk | one searchable passage; may span pages (`page` → `page_end`) |
| `Passage` | search | a chunk plus its similarity `score` for one question |

In [2]:
show(types.PageRecord, types.Chunk, types.Passage)

```python
@dataclass(frozen=True)
class PageRecord:
    """The text of one PDF page or one slide, before it is cut into chunks."""
    file: str
    page: int
    text: str
    kind: Kind
```

```python
@dataclass(frozen=True)
class Chunk:
    """One searchable passage: the unit that is embedded, indexed and cited.

    A passage from a PDF can run across a page break, so it records both the
    first and last page it covers. For slides, page and page_end are the same.
    """
    id: str
    file: str
    page: int
    page_end: int
    text: str
    kind: Kind

    def citation(self) -> str:
        """The label shown to the student, e.g. 'Lecture3.pdf, page 12' or 'Week5.pptx, slides 4–6'."""
        if self.page == self.page_end:
            return f"{self.file}, {self.kind} {self.page}"
        return f"{self.file}, {self.kind}s {self.page}–{self.page_end}"
```

```python
@dataclass(frozen=True)
class Passage(Chunk):
    """A chunk returned by search, with how well it matched the question."""
    score: float = 0.0
```

## 1 · Extract: pull the text out of each file
* **PDF** (PyMuPDF): one record per page. A PDF whose pages are mostly **landscape** is a slide deck saved as
  PDF, so its pages are marked `kind="slide"`.
* **PowerPoint** (python-pptx): a slide is a tree of *shapes* (title, text boxes, groups, tables), so
  `_shape_text` walks the tree and collects the text; speaker notes are added too.
* Anything else is rejected with a message the student can read.
* `file_sha256` fingerprints the file's bytes; it becomes the start of every chunk ID.

In [3]:
show(extract.extract, extract._extract_pdf, extract._looks_like_slides,
     extract._extract_pptx, extract._shape_text, extract.file_sha256)

```python
def extract(path,max_file_mb: int=50) -> list[PageRecord]:
    """Return the text of each page or slide, or raise FileRejected."""
    path=Path(path)
    suffix=path.suffix.lower()

    if suffix not in (".pdf",".pptx"):
        raise FileRejected(f"{path.name}: only PowerPoint (.pptx) and PDF files are supported.")

    size_mb = path.stat().st_size / (1024 * 1024)
    if size_mb > max_file_mb:
        raise FileRejected(f"{path.name} is {size_mb:.0f} MB; the limit is {max_file_mb} MB.")

    if suffix == ".pdf":
        return _extract_pdf(path)
    
    return _extract_pptx(path)
```

```python
def _extract_pdf(path: Path) -> list[PageRecord]:
    try:
        doc=pymupdf.open(path)
    except Exception as e:
        raise FileRejected(f"{path.name} could not be opened as a PDF.") from e

    with doc:
        kind = "slide" if _looks_like_slides(doc) else "page"
        pages = []
        for number, page in enumerate(doc, start=1):
            text = page.get_text()
            pages.append(PageRecord(file=path.name, page=number, text=text, kind=kind))
    return pages
```

```python
def _looks_like_slides(doc) -> bool:
    """A PDF counts as a slide deck if most of its pages are landscape."""
    landscape = sum(1 for page in doc if page.rect.width > page.rect.height)
    return landscape > len(doc) / 2
```

```python
def _extract_pptx(path: Path) -> list[PageRecord]:
    try:
        prs = Presentation(str(path))
    except Exception as e:
        raise FileRejected(f"{path.name} could not be opened as a PowerPoint file.") from e

    pages = []
    for number, slide in enumerate(prs.slides, start=1):
        parts = []
        for shape in slide.shapes:
            parts.extend(_shape_text(shape))

        if slide.has_notes_slide:
            notes = slide.notes_slide.notes_text_frame.text.strip()
            if notes:
                parts.append(f"Speaker notes: {notes}")

        pages.append(PageRecord(file=path.name, page=number, text="\n\n".join(parts), kind="slide"))
    return pages
```

```python
def _shape_text(shape) -> list[str]:
    """All text inside one shape, including groups and tables."""
    if shape.shape_type == MSO_SHAPE_TYPE.GROUP:
        texts = []
        for inner in shape.shapes:
            texts.extend(_shape_text(inner))
        return texts

    if shape.has_text_frame:
        text = shape.text_frame.text.strip()
        return [text] if text else []

    if shape.has_table:
        rows = []
        for row in shape.table.rows:
            rows.append(" | ".join(cell.text.strip() for cell in row.cells))
        return ["\n".join(rows)]

    return []
```

```python
def file_sha256(path) -> str:
    """A fingerprint of the file's contents: the same bytes always give the same hash."""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()
```

In [4]:
raw = {f.name: extract_file(f) for f in FILES}
for name, pages in raw.items():
    print(f"{name}: {len(pages)} records, kind={pages[0].kind}")

print("\nRaw text of Topic 8, slide 6:\n")
print(raw["Topic8_AgenticRAG_MITB.pptx"][5].text)

Topic7_Agentic_IS469.pdf: 65 records, kind=slide
Topic8_AgenticRAG_MITB.pptx: 54 records, kind=slide

Raw text of Topic 8, slide 6:

§1 · DEFINITION

What is RAG?

▸  RAG lets an LLM use user-designated external data sources as context to answer a query
▸  It combines a "retriever" of information with a "generator" of output
▸  The retriever fetches data, from the Internet or (usually) a vector database, and hands it, with the original query, to the generator
▸  The generator uses the retrieved data to produce its response
▸  Helps mitigate, though not eliminate, hallucination and bias, reducing many LLM ethical issues

Topic 8, Agentic RAG

MITB CS614


## 2 · Clean: tidy the text before chunking (Topic 1)
* **Normalise characters** (Unicode NFKC): ligatures like `ﬁ` → `fi`, non-breaking spaces → spaces, and
  invisible characters removed.
* **Drop repeated headers/footers**: a short line on at least half the pages (e.g. `Topic 8, Agentic RAG`)
  is boilerplate. Left in, it would make keyword search match *every* slide.
* **Drop page numbers**, but only on a page's first or last line, so list markers like `01` survive.
* **Rejoin wrapped lines** in document PDFs (not slides, where each line is a bullet).

In [5]:
show(clean.clean_pages, clean._strip_repeated_lines, clean._join_wrapped_lines)

```python
def clean_pages(pages: list[PageRecord]) -> list[PageRecord]:
    """Normalise characters, drop repeated headers/footers and page numbers,
    rejoin wrapped lines in documents, and drop pages left empty.
    Case is kept as-is; BM25 lowercases its own tokens.
    """
    pages = [replace(p, text=_normalize(p.text)) for p in pages]
    pages = _strip_repeated_lines(pages)
    cleaned = []
    for p in pages:
        text = p.text
        # Slide line breaks are bullets; PDF page line breaks are mostly just wrapping.
        if p.kind == "page":
            text = _join_wrapped_lines(text)
        text = _tidy_spaces(text)
        if text:
            cleaned.append(replace(p, text=text))
    return cleaned
```

```python
def _strip_repeated_lines(pages: list[PageRecord]) -> list[PageRecord]:
    """Remove short lines that appear on at least half the pages, and page numbers.
    A bare number only counts as a page number on a page's first or last line;
    elsewhere it is usually a list marker ("01  What RAG is") and is kept.
    """
    counts = Counter()
    for p in pages:
        counts.update({line.strip() for line in p.text.split("\n") if _is_short(line)})

    threshold = max(3, len(pages) // 2)
    repeated = {line for line, n in counts.items() if n >= threshold}

    stripped = []
    for p in pages:
        lines = [line for line in p.text.split("\n") if line.strip() not in repeated]
        while lines and not lines[0].strip():
            lines.pop(0)
        while lines and not lines[-1].strip():
            lines.pop()
        if lines and _PAGE_NUMBER.match(lines[-1].strip()):
            lines.pop()
        if lines and _PAGE_NUMBER.match(lines[0].strip()):
            lines.pop(0)
        stripped.append(replace(p, text="\n".join(lines)))
    return stripped
```

```python
def _join_wrapped_lines(text: str) -> str:
    # Undo words split across a line break ("informa-\ntion"). A real hyphenated
    # word that happens to break at its hyphen gets merged too; that trade-off is accepted.
    text = re.sub(r"(\w)-\n([a-z])", r"\1\2", text)
    out: list[str] = []
    for line in text.split("\n"):
        line = line.strip()
        if not line:
            out.append("")  # keep paragraph breaks
        elif out and out[-1] and not _BULLET.match(line):
            out[-1] += " " + line  # continuation of the previous line
        else:
            out.append(line)
    return "\n".join(out)
```

In [6]:
pages = {name: clean_pages(p) for name, p in raw.items()}

for name in raw:
    before = Counter(l.strip() for p in raw[name] for l in p.text.split("\n") if l.strip())
    after = Counter(l.strip() for p in pages[name] for l in p.text.split("\n") if l.strip())
    print(f"{name}: removed {(before - after).most_common(2)}")

print("\nTopic 8, slide 6 after cleaning (footer gone):\n")
print(pages["Topic8_AgenticRAG_MITB.pptx"][5].text)

Topic7_Agentic_IS469.pdf: removed [('Topic 7, Agentic AI', 53), ('IS469', 53)]
Topic8_AgenticRAG_MITB.pptx: removed [('Topic 8, Agentic RAG', 44), ('MITB CS614', 44)]

Topic 8, slide 6 after cleaning (footer gone):

§1 · DEFINITION

What is RAG?

▸ RAG lets an LLM use user-designated external data sources as context to answer a query
▸ It combines a "retriever" of information with a "generator" of output
▸ The retriever fetches data, from the Internet or (usually) a vector database, and hands it, with the original query, to the generator
▸ The generator uses the retrieved data to produce its response
▸ Helps mitigate, though not eliminate, hallucination and bias, reducing many LLM ethical issues


## 3 · Chunk: cut the text into passages
The chunk is the unit that is embedded, retrieved and cited, so **where we cut decides what search can find**
(slide 19, failure A: "chunking splits the answer so neither piece is retrieved"). Four methods:

| Method | Where it cuts | Idea |
|:--|:--|:--|
| **slide** | at every slide | the author already split the deck into one idea per slide |
| **fixed** *(baseline)* | every N words, with overlap | the method from the course lab (Lab B) |
| **recursive** | between paragraphs, then sentences; packs whole pieces up to N words | never cut mid-sentence |
| **semantic** | where neighbouring sentences change topic (embedding distance), max N words | cut at topic shifts (slide 22) |

Sizes are counted in **words**. Each word keeps its page number, so a chunk that crosses a page break cites
both pages (`slides 9–12`).

`chunk_pages` is the one entry point. Slide decks default to one chunk per slide, but `slide_method` can be
any other method, so the per-slide assumption is **tested**, not assumed.

In [7]:
show(chunk.chunk_pages, chunk._chunk_slides)

```python
def chunk_pages(pages: list[PageRecord], file_hash: str, chunk_size: int = 200,
                chunk_overlap: int = 30, method: str = "fixed", slide_method: str = "slide",
                embed: EmbedFn | None = None, semantic_percentile: float = 85) -> list[Chunk]:
    """Cut one file's cleaned pages into chunks.

    Documents use `method`; slide decks use `slide_method`, which can also be any
    document method (e.g. "recursive" packs small neighbouring slides together).
    Semantic chunking needs `embed`, and ignores chunk_overlap.
    """
    if not pages:
        return []

    if pages[0].kind == "slide":
        if slide_method == "slide":
            return _chunk_slides(pages, file_hash)
        method = slide_method

    if method in ("fixed", "recursive") and not 0 <= chunk_overlap < chunk_size:
        raise ValueError("chunk_overlap must be at least 0 and smaller than chunk_size")
    if method == "fixed":
        return _chunk_fixed(pages, file_hash, chunk_size, chunk_overlap)
    if method == "recursive":
        return _chunk_recursive(pages, file_hash, chunk_size, chunk_overlap)
    if method == "semantic":
        if embed is None:
            raise ValueError("semantic chunking needs an embed function")
        return _chunk_semantic(pages, file_hash, chunk_size, embed, semantic_percentile)
    raise ValueError(f"unknown chunking method: {method!r}")
```

```python
def _chunk_slides(pages: list[PageRecord], file_hash: str) -> list[Chunk]:
    """A slide is usually one idea, so each slide becomes exactly one chunk."""
    chunks = []
    for n, p in enumerate(pages):
        chunks.append(
            Chunk(
                id=_chunk_id(file_hash, n),
                file=p.file,
                page=p.page,
                page_end=p.page,
                text=p.text,
                kind="slide",
            )
        )
    return chunks
```

### Fixed windows (the baseline)
```
word      LoRA   freezes   the   weights   and   trains   small   matrices
page      1      1         1     1         2     2        2       2
chunk 0   ■      ■         ■     ■
chunk 1                          ■         ■     ■        ■           ← "weights" is the overlap
chunk 2                                                   ■       ■
```
Size 4, overlap 1: each window starts `size − overlap = 3` words after the last one.

In [8]:
show(chunk._chunk_fixed)

demo = [types.PageRecord("notes.pdf", 1, "LoRA freezes the weights", "page"),
        types.PageRecord("notes.pdf", 2, "and trains small matrices", "page")]
for c in chunk_pages(demo, "demo-hash-000", chunk_size=4, chunk_overlap=1, method="fixed"):
    print(f"{c.citation():22s} | {c.text}")

```python
def _chunk_fixed(pages, file_hash, size, overlap) -> list[Chunk]:
    """Cut every `size` words, each window starting `overlap` words before the last one ended."""
    words = [(word, p.page) for p in pages for word in p.text.split()]
    step = size - overlap
    chunks = []
    for start in range(0, len(words), step):
        window = words[start:start + size]
        text = " ".join(word for word, _ in window)
        chunks.append(_doc_chunk(pages, file_hash, len(chunks), text,
                                 first_page=window[0][1], last_page=window[-1][1]))
        if start + size >= len(words):
            break
    return chunks
```

notes.pdf, page 1      | LoRA freezes the weights
notes.pdf, pages 1–2   | weights and trains small
notes.pdf, page 2      | small matrices


### Recursive: split into pieces that fit, then pack whole pieces
1. **Split** each paragraph into pieces of at most N words: the whole paragraph if it fits, otherwise its
   sentences, otherwise (last resort) word windows.
2. **Pack** pieces into a chunk until the next one would not fit; then start a new chunk, carrying over the
   last whole sentence(s) as overlap. Every chunk starts and ends on a sentence boundary.

In [9]:
show(chunk._split_to_fit, chunk._chunk_recursive, chunk._overlap_tail)

demo = [types.PageRecord("notes.pdf", 1, "LoRA is a cheap fine-tuning method.\n\nIt freezes the original weights.", "page"),
        types.PageRecord("notes.pdf", 2, "It trains two small matrices called A and B. They are added to each layer.", "page")]
for c in chunk_pages(demo, "demo-hash-000", chunk_size=14, chunk_overlap=6, method="recursive"):
    print(f"{c.citation():22s} | {c.text}")

```python
def _split_to_fit(text: str, size: int) -> list[str]:
    """Split one paragraph into pieces of at most `size` words: whole paragraph if it
    fits, otherwise sentences, otherwise fixed word windows."""
    if len(text.split()) <= size:
        return [text.strip()] if text.strip() else []
    pieces = []
    for sentence in _SENTENCE_END.split(text):
        words = sentence.split()
        if not words:
            continue
        if len(words) <= size:
            pieces.append(" ".join(words))
        else:
            for i in range(0, len(words), size):
                pieces.append(" ".join(words[i:i + size]))
    return pieces
```

```python
def _chunk_recursive(pages, file_hash, size, overlap) -> list[Chunk]:
    """Pack whole paragraphs (or sentences) into chunks of at most `size` words."""
    # 1. Split: every piece fits within `size`, and remembers its page.
    pieces = []  # (text, page)
    for p in pages:
        for para in re.split(r"\n\s*\n", p.text):
            for piece in _split_to_fit(para, size):
                pieces.append((piece, p.page))

    # 2. Pack: add pieces until the next one won't fit, then start a new chunk.
    chunks = []
    current = []
    for piece in pieces:
        if current and _n_words(current) + _n_words([piece]) > size:
            chunks.append(_pieces_to_chunk(pages, file_hash, len(chunks), current))
            current = _overlap_tail(current, overlap)
            while current and _n_words(current) + _n_words([piece]) > size:
                current.pop(0)
        current.append(piece)
    if current:
        chunks.append(_pieces_to_chunk(pages, file_hash, len(chunks), current))
    return chunks
```

```python
def _overlap_tail(pieces, overlap):
    """The last few whole pieces of a chunk, totalling at most `overlap` words."""
    tail = []
    for piece in reversed(pieces):
        if _n_words(tail) + _n_words([piece]) > overlap:
            break
        tail.insert(0, piece)
    return tail
```

notes.pdf, page 1      | LoRA is a cheap fine-tuning method. It freezes the original weights.
notes.pdf, pages 1–2   | It freezes the original weights. It trains two small matrices called A and B.
notes.pdf, page 2      | They are added to each layer.


### Semantic: cut where the topic changes
1. Split into sentences (on slides, each bullet line counts as one).
2. Embed every sentence and measure the **gap** between neighbours: `1 − cosine similarity`.
3. Cut where a gap is in the top 15% for this file (the 85th percentile), so the threshold adapts to each
   document; also cut when the chunk is full. A chunk is never cut for a topic shift before it holds a fifth of
   N words, so stray lines like `01` don't become chunks of their own.

In [10]:
show(chunk._chunk_semantic)

```python
def _chunk_semantic(pages, file_hash, size, embed: EmbedFn, percentile: float) -> list[Chunk]:
    """Start a new chunk where the next sentence's meaning shifts sharply from the
    current one, or when the chunk would exceed `size` words.

    A "sharp shift" is a gap bigger than `percentile` percent of all the gaps in
    this file, so the cut-off adapts to each document. A chunk is never cut for a
    topic shift before it holds a fifth of `size`, so stray lines like "01" don't
    become chunks of their own.
    """
    min_words = size // 5
    sentences = _sentences(pages, size)
    if not sentences:
        return []

    # 1. Embed every sentence; normalise so a dot product is the cosine similarity.
    vectors = np.asarray(embed([text for text, _ in sentences]), dtype=float)
    vectors /= np.linalg.norm(vectors, axis=1, keepdims=True).clip(min=1e-12)

    # 2. gaps[i] = how different sentence i+1 is from sentence i (0 = same meaning).
    gaps = 1 - np.sum(vectors[:-1] * vectors[1:], axis=1)
    threshold = np.percentile(gaps, percentile) if len(gaps) else 0.0

    # 3. Walk through the sentences, cutting at topic shifts or when the chunk is full.
    chunks = []
    current = [sentences[0]]
    for i, sentence in enumerate(sentences[1:]):
        topic_shift = gaps[i] > threshold and _n_words(current) >= min_words
        too_big = _n_words(current) + _n_words([sentence]) > size
        if topic_shift or too_big:
            chunks.append(_pieces_to_chunk(pages, file_hash, len(chunks), current))
            current = []
        current.append(sentence)
    chunks.append(_pieces_to_chunk(pages, file_hash, len(chunks), current))
    return chunks
```

## 4 · Embed: turn each passage into a vector
`all-MiniLM-L6-v2` (the model from the course labs) maps text to 384 numbers so that passages with similar
meaning land close together. Vectors are normalised to length 1, so a dot product **is** the cosine similarity.

⚠️ The model reads at most **256 tokens** per text and silently ignores the rest, which matters for big chunks.

In [11]:
show(Embedder)

embed = Embedder()
v = embed(["Re-ranking reorders the retrieved chunks with a cross-encoder.",
           "A second model re-scores the candidate passages together with the question.",
           "MCP servers expose tools, resources and prompts."])
print("vector length:", v.shape[1], "| reads at most", embed.max_tokens, "tokens")
print("same idea, different words:", round(float(v[0] @ v[1]), 3))
print("different topic:           ", round(float(v[0] @ v[2]), 3))

```python
class Embedder:
    """Callable wrapper: embedder(["text", ...]) -> array with one row per text.

    Vectors are normalised to length 1, so a dot product between two of them is
    their cosine similarity.
    """

    def __init__(self, model_name: str = DEFAULT_MODEL):
        self.model_name = model_name
        self.model = _load(model_name)

    @property
    def max_tokens(self) -> int:
        """Tokens the model reads per text; anything longer is silently cut off."""
        return self.model.max_seq_length

    def __call__(self, texts: list[str]) -> np.ndarray:
        return self.model.encode(texts, normalize_embeddings=True, convert_to_numpy=True,
                                 show_progress_bar=False)
```

/Users/yanting/Desktop/IS-469/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1022 23:43:14.549000 77629 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7185.13it/s]

vector length: 384 | reads at most 256 tokens
same idea, different words: 0.357
different topic:            0.001


Now chunk one deck with every method (size 200 words) and compare what they produce:

In [12]:
hashes = {f.name: file_sha256(f) for f in FILES}
deck = "Topic8_AgenticRAG_MITB.pptx"

for m in ("slide", "fixed", "recursive", "semantic"):
    chunks = chunk_pages(pages[deck], hashes[deck], chunk_size=200, chunk_overlap=30,
                         slide_method=m, embed=embed)
    words = [len(c.text.split()) for c in chunks]
    print(f"{m:10s} {len(chunks):3d} chunks | words min {min(words):3d} avg {sum(words)//len(words):3d} "
          f"max {max(words):3d} | e.g. {chunks[3].citation()}")

slide       54 chunks | words min  12 avg  63 max 113 | e.g. Topic8_AgenticRAG_MITB.pptx, slide 4
fixed       20 chunks | words min 192 avg 199 max 200 | e.g. Topic8_AgenticRAG_MITB.pptx, slides 10–13
recursive   21 chunks | words min 122 avg 177 max 200 | e.g. Topic8_AgenticRAG_MITB.pptx, slides 9–12


semantic    31 chunks | words min  40 avg 110 max 200 | e.g. Topic8_AgenticRAG_MITB.pptx, slides 4–5


## 5 · Search: rank passages by cosine similarity
Index once (embed every chunk), then for each question: embed it with the **same** model, compare it with
every chunk at once (`vectors @ q`), and return the top-k. Brute force is exact and fast at this scale
(slide 17). Search runs over **all files** together, like a student's notebook.

In [13]:
show(DenseIndex)

```python
class DenseIndex:
    """Holds one vector per chunk and returns the top-k chunks for a question."""

    def __init__(self, chunks: list[Chunk], embed: EmbedFn):
        self.chunks = chunks
        self.embed = embed
        self.vectors = _normalise(embed([c.text for c in chunks])) if chunks else np.zeros((0, 1))

    def search(self, query: str, k: int = 5) -> list[Passage]:
        if not self.chunks:
            return []
        q = _normalise(self.embed([query]))[0]
        scores = self.vectors @ q  # cosine similarity with every chunk at once
        top = np.argsort(-scores)[:k]
        return [Passage(**asdict(self.chunks[i]), score=float(scores[i])) for i in top]
```

In [14]:
all_chunks = [c for f in FILES for c in chunk_pages(pages[f.name], hashes[f.name])]  # one chunk per slide
index = DenseIndex(all_chunks, embed)

# ▶ Try your own question here
QUESTION = "Why does a cross-encoder rank better than cosine similarity?"
print("Q:", QUESTION, "\n")
for rank, p in enumerate(index.search(QUESTION), start=1):
    print(f"{rank}. {p.score:.3f}  {p.citation()}")
    print("   ", p.text[:150].replace("\n", " "), "...\n")

Q: Why does a cross-encoder rank better than cosine similarity? 

1. 0.591  Topic8_AgenticRAG_MITB.pptx, slide 32
    §5 · ENCODERS  Bi-encoder vs. cross-encoder  ADVANCED  Bi-encoder (the vector DB)  • Chunk and query embedded separately • Embeddings precomputed, que ...

2. 0.570  Topic8_AgenticRAG_MITB.pptx, slide 33
    §5 · IN PRACTICE  Re-rankers in practice  ADVANCED  ▸ BERT-type cross-encoders do the re-ranking, also sold as APIs ▸ Cohere Rerank 4: rerank-v4.0-fas ...

3. 0.523  Topic8_AgenticRAG_MITB.pptx, slide 31
    §5 · WHY IT HELPS  Why re-ranking beats plain cosine  ▸ The vector DB embedded each chunk when it was loaded, before any query existed ▸ Those chunk e ...

4. 0.460  Topic8_AgenticRAG_MITB.pptx, slide 15
    §2 · VECTOR SEARCH  What 'retrieve top-k by cosine' actually means  Illustrative: query and document chunks are points in embedding space; retrieval r ...

5. 0.378  Topic8_AgenticRAG_MITB.pptx, slide 54
    LAB  Hands-on lab & what's expected  ▸ Lab A, Topi

## 6 · Evaluate: which chunking method finds the right slides?
**Gold set** (`eval/gold/slides.csv`): 40 questions (20 per deck), each labelled with the slide(s) holding the
answer. In `gold_slides`, `;` lists alternative slides that each hold the whole answer, and `|` separates the
parts of a multi-part question. A retrieved chunk counts as a hit if its slide range covers a gold slide.

| Metric | Question it answers |
|:--|:--|
| **Hit@5** | Was at least one right slide in the top 5? |
| **Recall@5** | What share of the answer's parts were found in the top 5? (matters for multi-part questions) |
| **MRR** | How high was the first right slide? (1 at rank 1, ½ at rank 2, … 0 if missing) |

Everything else is held fixed (same embedder, dense search, k = 5), so differences come from chunking alone.
The scoring code is shared with `eval/eval_retrieval.py`, the script version of this section.

In [15]:
import eval_retrieval as ev

show(ev.load_gold, ev.score)

questions = ev.load_gold(ROOT / "eval" / "gold" / "slides.csv")
print(len(questions), "gold questions, e.g.:", questions[0]["question"], "→ slides", questions[0]["gold_slides"])

```python
def load_gold(path: Path) -> list[dict]:
    with open(path, newline="") as f:
        questions = list(csv.DictReader(f))
    for q in questions:
        q["parts"] = [{int(s) for s in part.split(";")} for part in q["gold_slides"].split("|")]
    return questions
```

```python
def score(question: dict, passages) -> dict:
    """Which parts of the answer were found in the top-k, and the rank of the first hit."""

    def answers(passage, part) -> bool:
        return passage.file == question["file"] and any(passage.page <= s <= passage.page_end for s in part)

    found = sum(any(answers(p, part) for p in passages) for part in question["parts"])
    first = next((rank for rank, p in enumerate(passages, start=1)
                  if any(answers(p, part) for part in question["parts"])), None)
    return {
        "hit": int(found > 0),
        "recall": found / len(question["parts"]),
        "rr": 1 / first if first else 0.0,
        "retrieved": " ; ".join(p.citation() for p in passages),
    }
```

40 gold questions, e.g.: What two components does retrieval-augmented generation combine? → slides 6;53


In [16]:
K = 5
setups = list(ev.setups(is_slides=True))  # slide, then fixed/recursive/semantic at 100/200/400 words
results, misses = [], Counter()

for name, settings in setups:
    chunks = [c for f in FILES for c in chunk_pages(pages[f.name], hashes[f.name], embed=embed, **settings)]
    index = DenseIndex(chunks, embed)
    scores = [ev.score(q, index.search(q["question"], K)) for q in questions]
    misses.update(q["id"] for q, s in zip(questions, scores) if not s["hit"])
    over = np.mean([ev.n_tokens(embed, c.text) > embed.max_tokens for c in chunks])
    results.append((name, np.mean([s["hit"] for s in scores]), np.mean([s["recall"] for s in scores]),
                    np.mean([s["rr"] for s in scores]), len(chunks), over))

print(f"{'setup':15s} {'Hit@5':>6s} {'Recall@5':>9s} {'MRR':>5s} {'chunks':>7s} {'>256 tok':>9s}")
for name, hit, rec, mrr, n, over in results:
    print(f"{name:15s} {hit:6.2f} {rec:9.2f} {mrr:5.2f} {n:7d} {over:9.0%}")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (276 > 256). Running this sequence through the model will result in indexing errors


setup            Hit@5  Recall@5   MRR  chunks  >256 tok
slide             0.93      0.91  0.77     119        1%
fixed-100         0.85      0.85  0.73      92        0%
fixed-200         0.88      0.85  0.75      45       84%
fixed-400         0.85      0.84  0.68      23      100%
recursive-100     0.90      0.90  0.80     102        0%
recursive-200     0.90      0.89  0.79      50       32%
recursive-400     0.88      0.88  0.72      23       96%
semantic-100      0.90      0.89  0.69     120        0%
semantic-200      0.93      0.91  0.72      80       12%
semantic-400      0.95      0.93  0.76      50       24%


### Which questions are hard for every setup?
A question missed by almost every setup is a **search** problem, not a chunking one, and points to the next
improvement.

In [17]:
by_id = {q["id"]: q for q in questions}
for qid, n in misses.most_common(5):
    print(f"missed in {n}/{len(setups)} setups | {qid}: {by_id[qid]['question']} (gold slides {by_id[qid]['gold_slides']})")

missed in 10/10 setups | t8-14: What is HyDE? (gold slides 39)
missed in 7/10 setups | t8-01: What two components does retrieval-augmented generation combine? (gold slides 6;53)
missed in 7/10 setups | t8-16: What changes about control flow when the agent is the one deciding when to retrieve? (gold slides 44;46)
missed in 4/10 setups | t8-13: How does contextual RAG help a chunk that loses its meaning once it is taken out of the document? (gold slides 36;37)
missed in 4/10 setups | t7-18: When is it a mistake to use an agent instead of ordinary code? (gold slides 49)


## Findings
* **Fixed windows (the lab's method) are the weakest at every size.** Cutting blindly through slides mixes
  topics.
* **Per-slide and semantic chunking find the answer most often**; **recursive ranks it highest** (best MRR).
  With 40 questions, one question = 2.5 points, so gaps of 1–2 questions between these three are roughly a tie.
* **Big chunks are silently truncated by the embedder**: most 200- and 400-word fixed/recursive chunks exceed
  its 256-token window, and their scores drop.
* **"What is HyDE?" fails in every setup**: an acronym the embedder doesn't understand. That is the exact-term
  weakness from slide 35 (and Lab B's "KN-4471"), and the reason to add **BM25 keyword search** next.

**Next steps:** hybrid search (BM25 + RRF) and cross-encoder re-ranking, compared on the same gold set; then
the prose and rules corpora.